# PDEFlow · observation-informed physical mesh models

M1 deterministic initial reconstruction → M2 learned evolution → M3 FGN-style → matched M4 flow / M5 diffusion. No mandatory autoencoder-code target in these new models. The original A1/A2/B notebook is preserved as `legacy-factorized-colab.ipynb`.

This notebook only calls shared package code. Choose one candidate or an explicit suite; preview, preflight, then start deliberately. New runs use MSE and known continuous measurement timestamps. Region-balanced and spectral objectives remain explicit experiments. Pipeline checks are not accuracy evidence. No CUDA/Colab execution has been claimed by the local handoff.

For a new experiment use the new source ZIP in a fresh runtime. For resume reload that run's exact saved `source/source.zip` first. Never install new scientific code under an active training process.

In [ ]:
#@title 1 · Set up project (archive or private/public checkout)
from pathlib import Path
import os, sys, subprocess, zipfile, hashlib, json, tempfile, urllib.parse
REPOSITORY_URL = "" #@param {type:"string"}
REVISION = "" #@param {type:"string"}
GITHUB_SECRET_NAME = "" #@param {type:"string"}
SOURCE_ARCHIVE = "" #@param {type:"string"}
CACHE_ROOT = Path("/content/pdeflow-source") # disposable; configurable
INSTALL_PROJECT = True #@param {type:"boolean"}
EXPECTED_SOURCE_SHA256 = "" # optional handoff hash, never inferred from a model
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
if not REPOSITORY_URL and not SOURCE_ARCHIVE:
    from google.colab import files
    uploaded = files.upload()  # upload the source ZIP, not model weights
    if len(uploaded) != 1: raise ValueError("Upload exactly one source ZIP")
    SOURCE_ARCHIVE = next(iter(uploaded))
if REPOSITORY_URL:
    parsed = urllib.parse.urlsplit(REPOSITORY_URL)
    if parsed.username or parsed.password or parsed.query or parsed.fragment: raise ValueError("Do not put tokens in a URL; use a runtime secret name")
    if not REVISION: raise ValueError("Set an exact checkout revision for reproducible training")
    source_key = hashlib.sha256((REPOSITORY_URL + REVISION).encode()).hexdigest()
    PACKAGE_ROOT = CACHE_ROOT / source_key[:16]
    if not PACKAGE_ROOT.exists():
        env = os.environ.copy()
        with tempfile.TemporaryDirectory() as auth:
            if GITHUB_SECRET_NAME:
                from google.colab import userdata
                token = userdata.get(GITHUB_SECRET_NAME)
                ask = Path(auth) / "askpass.py"
                ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["PDEFLOW_GIT_TOKEN"])\n')
                ask.chmod(0o700); env.update(GIT_ASKPASS=str(ask), PDEFLOW_GIT_TOKEN=token, GIT_TERMINAL_PROMPT="0")
            for cmd in (["git", "clone", "--no-checkout", REPOSITORY_URL, str(PACKAGE_ROOT)], ["git", "-C", str(PACKAGE_ROOT), "checkout", "--detach", REVISION]):
                result = subprocess.run(cmd, env=env, capture_output=True, text=True)
                if result.returncode: raise RuntimeError("Checkout failed; check revision/access. Credentials are not printed.")
    revision = subprocess.check_output(["git", "-C", str(PACKAGE_ROOT), "rev-parse", "HEAD"], text=True).strip()
    print("Source revision:", revision)
else:
    archive = Path(SOURCE_ARCHIVE).expanduser().resolve()
    source_key = hashlib.sha256(archive.read_bytes()).hexdigest()
    if EXPECTED_SOURCE_SHA256 and source_key != EXPECTED_SOURCE_SHA256: raise ValueError("Unexpected source archive identity")
    extracted = CACHE_ROOT / source_key[:16]
    if not extracted.exists():
        with zipfile.ZipFile(archive) as z:
            if sum(i.file_size for i in z.infolist()) > 512 * 1024**2: raise ValueError("Source ZIP exceeds 512 MiB unpacked")
            for i in z.infolist():
                p = Path(i.filename)
                if p.is_absolute() or ".." in p.parts or (i.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError("Unsafe archive member")
            z.extractall(extracted)
    roots = [p.parent for p in extracted.rglob("pyproject.toml") if (p.parent / "src/pdeflow").is_dir()]
    if len(roots) != 1: raise ValueError("ZIP must contain exactly one PDEFlow source root")
    PACKAGE_ROOT = roots[0]
    # Rerun safety: verify cached members rather than trusting a stale extraction.
    with zipfile.ZipFile(archive) as z:
        for i in z.infolist():
            member = Path(i.filename)
            if member.is_absolute() or ".." in member.parts or (i.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError("Unsafe archive member")
            if i.is_dir(): continue
            cached = extracted / member
            if cached.is_symlink() or not cached.resolve().is_relative_to(extracted.resolve()) or not cached.is_file() or hashlib.sha256(cached.read_bytes()).digest() != hashlib.sha256(z.read(i)).digest():
                raise ValueError("Cached source changed; choose a fresh CACHE_ROOT. Existing files are not overwritten.")
    print("Source archive SHA256:", source_key)
if "pdeflow" in sys.modules and not Path(sys.modules["pdeflow"].__file__).resolve().is_relative_to(PACKAGE_ROOT.resolve()):
    raise RuntimeError("Restart the kernel before changing source. Do not modify source under active training.")
subprocess.run([sys.executable, str(PACKAGE_ROOT / "scripts/colab_setup.py"), *([] if INSTALL_PROJECT else ["--check-only"])], check=True)
# Editable .pth files installed by subprocess pip are not reloaded in this kernel.
if "pdeflow" in sys.modules and not Path(sys.modules["pdeflow"].__file__).resolve().is_relative_to(PACKAGE_ROOT.resolve()):
    raise RuntimeError("Source changed in this running kernel. Restart, then select the intended source again.")
sys.path.insert(0, str(PACKAGE_ROOT / "src"))
import importlib
importlib.invalidate_caches()
# --no-deps setup retains the runtime's compatible PyTorch/CUDA stack.
# If verification fails, select a supported Python3.11/3.12 runtime (e.g. 2026.07).
# No automatic PyTorch replacement or hidden runtime restart occurs.

from pdeflow.source_bundle import verify as verify_source
if not REPOSITORY_URL: print("Verified source:", verify_source(archive, PACKAGE_ROOT))
from pdeflow.mesh.provenance import scientific_source
print("Scientific source identity:", scientific_source()["sha256"])


## 2 · Storage and recovery

Runtime disk is disposable. Verified content-addressed snapshots retain source, exact configurations, actual modules, optimizer/scheduler/scaler/RNG/counters and case/history metadata. First checkpoint, cadence, stage transitions, safe stop and final exports are persisted. A hard crash may lose work since the last verified snapshot. This is not an atomic Google Drive rename guarantee.

Keep the run ID printed below. An interrupted copy cannot replace the last complete verified snapshot. Restore into a fresh empty local root; existing conflicting files are not overwritten. Actual Drive mounting is a Colab operation, not locally verified.

In [ ]:
#@title 2 · Configure storage (no training)
ENABLE_DRIVE_PERSISTENCE = True #@param {type:"boolean"}
DRIVE_MOUNT = "/content/drive" #@param {type:"string"}
DRIVE_RUNS_ROOT = "/content/drive/MyDrive/PDEFlow-mesh" #@param {type:"string"}
LOCAL_ROOT = "/content/pdeflow-mesh" #@param {type:"string"}
RUN_MODE = "new" #@param ["new", "resume"]
RUN_ID = "" # required for resume; printed when creating a new run
SNAPSHOT_ID = "" # blank chooses latest verified complete snapshot
SAVE_EVERY_SUCCESSFUL_UPDATES = 100
KEEP_SNAPSHOTS = 3
if ENABLE_DRIVE_PERSISTENCE:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    if not Path(DRIVE_MOUNT).is_mount(): raise RuntimeError("Drive is not mounted; refusing disposable storage labeled Drive")
    from pdeflow.persistence import probe
    print("Storage probe:", probe(Path(DRIVE_RUNS_ROOT)/"probe-run"))
else:
    print("PERSISTENCE DISABLED: runtime loss can lose all local artifacts.")
Path(LOCAL_ROOT).mkdir(parents=True,exist_ok=True)
from pdeflow.persistence import snapshots
if RUN_MODE == "resume":
    if not RUN_ID: raise ValueError("Set the exact saved RUN_ID")
    print("Verified snapshots:",[(s['id'],s.get('sequence')) for s in snapshots(Path(DRIVE_RUNS_ROOT)/RUN_ID)])


## 3 · Candidate, capacity and science

M1 reconstructs only t=0, even when its sensors are later. M2 adds observation-only physical evolution. M3 uses central pretraining then independent two-draw fair CRPS, with separately weighted zero-noise supervision. M4/M5 first learn physical T/D from clean TRAIN initial nodes, then freeze them and learn conditional physical-state generation; this pretraining cost is recorded. No A1 encoder exists in these models.

Heat state is u; wave state is u and v; wave speed c enters the equation as c². Unknown coefficients are explicit positive generated inputs. Initial-field shape mixtures are not PDE families. Measurement times, physical target times and generative noise times are independent.

Change science only for NEW experiments. Resume uses recorded configuration/source and strict dependencies. Candidate order does not change training seeds/data. An explicit warm start is a new experiment with parent cost, not resume or a from-scratch comparison.

In [ ]:
#@title 3 · Resolve one candidate or suite (no training)
from pdeflow.mesh.contracts import preset,MeshConfig,CANDIDATES
from pdeflow.mesh.session import MeshSession
from pdeflow.mesh.settings import configure,summary
from pdeflow.mesh.suite import Suite,suite_plan
from pdeflow.runtime import read_json,write_json
CANDIDATE = "mesh_m1_v1" #@param ["mesh_m1_v1","mesh_m2_v1","mesh_m3_fgn_v1","mesh_m4_flow_v1","mesh_m5_diffusion_v1"]
CAPACITY = "medium" #@param ["pipeline-check","xsmall","small","medium","large","xlarge"]
DEVICE = "cuda" #@param ["cuda","mps","cpu"]
PRECISION = "auto" #@param ["auto","fp32","fp16","bf16"]
TRAINING_EQUATIONS = "Heat + Wave" #@param ["Heat only","Wave only","Heat + Wave"]
SUITE = False #@param {type:"boolean"}
INCLUDE_LEGACY = False # original trainer, additional independently recorded cost
FIELD_LOSS = "mse" #@param ["mse","weighted_mse","region_balanced_mse","mae","spectral_mse"]
FEATURE_BALANCE_ALPHA = 0.5
LEARNING_RATE = 0.001 #@param {type:"number"}
EFFECTIVE_BATCH = 8 #@param {type:"integer"}
FORWARD_MICROBATCH = 8 #@param {type:"integer"}
BATCH_BUDGET_POLICY = "keep_updates" #@param ["keep_updates","keep_examples"]
STAGE_OVERRIDES = {} # Advanced: {"stochastic":{"learning_rate":1e-4,"batch_size":8,"microbatch":4,"optimizer":{"warmup":100}}}
# Learning rate 1e-4 is a new-run experiment, not a proven instability fix.
# keep_examples rounds original per-stage example budgets UP; LR/warmup never auto-scale.
UPDATES_PER_STAGE = None # None retains capacity budget; successful optimizer updates
ADVANCED = {} # other MeshConfig overrides; optimizer/batch controls above take precedence
# e.g. {"objective":{"observation":0.001}}; no new stage gate is bypassed
persistent_root=Path(DRIVE_RUNS_ROOT) if ENABLE_DRIVE_PERSISTENCE else None
if RUN_MODE == "resume":
    print("RESUME: recorded optimizer/batch/stage settings retained; new-run controls above are ignored. Changed settings require a NEW experiment, not an implicit continuation.")
    if not ENABLE_DRIVE_PERSISTENCE: raise ValueError("Fresh-runtime restore needs a configured verified store; use MeshSession.reopen for existing local runs")
    session=Suite.from_store(persistent_root,RUN_ID,LOCAL_ROOT,SNAPSHOT_ID or None) if SUITE else MeshSession.from_store(persistent_root,RUN_ID,LOCAL_ROOT,SNAPSHOT_ID or None,device=DEVICE,save_updates=SAVE_EVERY_SUCCESSFUL_UPDATES,keep_snapshots=KEEP_SNAPSHOTS)
else:
    families={"heat":1.} if TRAINING_EQUATIONS=="Heat only" else {"wave":1.} if TRAINING_EQUATIONS=="Wave only" else {"heat":.5,"wave":.5}
    field={"type":FIELD_LOSS}
    if FIELD_LOSS=="region_balanced_mse":field["alpha"]=FEATURE_BALANCE_ALPHA
    if FIELD_LOSS=="spectral_mse":field["spectral_weight"]=0.1
    overrides={**ADVANCED,"family_weights":families,"field_loss":field}
    for controlled in ("learning_rate","batch_size","microbatch","stage_settings","batch_policy","batch_reference"):
        if controlled in ADVANCED:raise ValueError("Use explicit training controls, not ADVANCED: "+controlled)
    training_settings={"learning_rate":LEARNING_RATE,"effective_batch":EFFECTIVE_BATCH,"microbatch":FORWARD_MICROBATCH,"stages":STAGE_OVERRIDES,"budget_policy":BATCH_BUDGET_POLICY}
    if UPDATES_PER_STAGE is not None:overrides["budgets"]={s:UPDATES_PER_STAGE for s in MeshConfig().budgets}
    if SUITE:
        resolved=suite_plan(CAPACITY,DEVICE,PRECISION,overrides=overrides,include_legacy=INCLUDE_LEGACY,training_settings=training_settings)
        session=Suite.new(resolved,LOCAL_ROOT,persistent_root)
    else:
        config=configure(preset(CAPACITY,CANDIDATE,DEVICE,PRECISION,overrides),**training_settings)
        session=MeshSession.new(config,CANDIDATE,LOCAL_ROOT,persistent_root,save_updates=SAVE_EVERY_SUCCESSFUL_UPDATES,keep_snapshots=KEEP_SNAPSHOTS)
print("SAVE RUN ID:",session.root.name,"Local:",session.root,"Persistent:",session.persistent)
print(json.dumps(session.inspect(),indent=2))

resolved_optimizers={name:summary(MeshConfig(**cfg).validate()) for name,cfg in session.inspect()["plan"]["configs"].items()} if SUITE else summary(session.c)
print("RESOLVED ADAMW / BATCH / ACCUMULATION (before training):")
print(json.dumps(resolved_optimizers,indent=2))


## 4 · Actual-candidate preflight

Never starts automatically. Uses the same architecture, query/sensor counts, losses, batch and precision, with disposable short budgets. Reports setup/cold update, warm median/range/p90, examples/s, actual microbatch, reference/compute and CUDA peak memory. Includes at least five measured successful updates after warmup. Results are not accuracy evidence. Auto precision chooses native BF16 only when supported; T4 uses FP16+GradScaler. A reported successful-update skip is not a training update.

Substantial new runs require a matching successful preflight. Change settings in step3 only for a new run, then preflight again. A failed gate stops subsequent stages/suite; improve prerequisites before further substantial work.

In [ ]:
#@title 4 · Optional real preflight
RUN_PREFLIGHT = False
PREFLIGHT_CANDIDATE = "" # suite: exact architecture ID; blank explicitly profiles all
if RUN_PREFLIGHT:
    profile=session.preflight(updates=8,warmup=2,candidate=PREFLIGHT_CANDIDATE or None) if SUITE else session.preflight(updates=8,warmup=2)
    print(json.dumps(profile,indent=2))
else:
    print("Preflight not launched. Select it explicitly before substantial training.")

# Independent bounded sweep. Does NOT apply a winner or modify the Session.
RUN_MICROBATCH_BENCHMARK = False #@param {type:"boolean"}
BENCHMARK_MICROBATCHES = "1,2,4,8" #@param {type:"string"}
BENCHMARK_UPDATES = 8 # two warmup + six measured successful updates per stage
BENCHMARK_WARMUP = 2
if RUN_MICROBATCH_BENCHMARK:
    from pdeflow.mesh.benchmark import microbatch_benchmark
    from pdeflow.local import is_locked
    if is_locked(session.root):raise ValueError("Stop training before benchmarking")
    if SUITE:
        if PREFLIGHT_CANDIDATE not in session.inspect()["plan"]["configs"]:raise ValueError("Choose an exact mesh PREFLIGHT_CANDIDATE for a bounded sweep")
        benchmark_config=MeshConfig(**session.inspect()["plan"]["configs"][PREFLIGHT_CANDIDATE]).validate()
    else:benchmark_config=session.c
    candidates=[int(x.strip()) for x in BENCHMARK_MICROBATCHES.split(",")]
    result=microbatch_benchmark(benchmark_config,session.root/"benchmarks",candidates,BENCHMARK_UPDATES,BENCHMARK_WARMUP)
    for item in result["candidates"]:
        if item["state"]!="completed":
            print("FAILED",item["microbatch"],item["error"],item["traceback"]);continue
        for stage,row in item["report"]["stages"].items():
            print("microbatch",item["microbatch"],stage,"examples/s",row["examples_per_second"],"end-to-end s/update",row["end_to_end_seconds_per_successful_update"],"GPU peak bytes",row["memory"]["cuda_peak_allocated"],"successful/skipped",row["successful_updates"],row["skipped_updates"],"setup s",row["setup_seconds"])
        print("Estimated all-stage hours:",item["report"]["estimated_total_training_seconds"]/3600)
    print("Full report:",result["path"])
    print("No settings applied. For a NEW run edit Step 3 explicitly, then preflight the final configuration. Started runs retain recorded settings. To test 8,16,32 first explicitly set effective batch >=32 and choose keep_updates/keep_examples.")


## 5 · Inspect / safe stop

Review the exact plan and separate stage costs. Effective batch is total examples per successful update; microbatch controls batched forward/backward execution, not total examples. MSE is the default deterministic field loss. FGN fair CRPS, FM velocity and diffusion VP-v are distinct objectives; do not compare their raw loss values.

The primary training cell displays live progress every5 seconds, including setup, validation, checkpointing and verified Drive sync. A second cell in the same blocked kernel is NOT concurrent monitoring. Use a separate terminal/process for read-only status or a cooperative stop request; it is honored after an optimizer boundary and persists complete state. Runtime interrupt also cooperatively saves, but abrupt termination is limited to the latest verified snapshot. Reopening and running resumes incomplete stages and skips verified completed suite candidates.

In [ ]:
#@title 5 · Read-only status / cooperative stop (no training)
from pdeflow.mesh.monitoring import status,text_summary
REQUEST_SAFE_STOP = False
if REQUEST_SAFE_STOP:session.stop()
print(text_summary(status(session.root)))
print("A second cell cannot concurrently execute during blocking training in this kernel. Use runtime interrupt for safe stop, or a separate terminal/process: pdeflow mesh status --run ROOT --watch")


## 6 · Explicit train / resume / remaining suite

Off by default. Later stages are blocked on prerequisite failure for real experiments. Pipeline-check configs are explicit execution diagnostics, not a successful scientific gate. There is no poor-quality override. The final state and full traceback are shown directly if a run fails. To continue after a runtime loss, use the same source, `RUN_MODE="resume"`, correct `RUN_ID` and `SUITE`, then steps2–3 and6.

In [ ]:
#@title 6 · Train only by explicit request
RUN_TRAINING = False
SELECTED_CANDIDATES = [] # suite: [] means remaining in recorded order
from pdeflow.mesh.monitoring import run as run_monitored
if RUN_TRAINING:
    try:
        outcome=run_monitored(session,refresh_seconds=5,plots=True,selected=SELECTED_CANDIDATES or None)
        print("Finished:",outcome["state"],"· records:",session.root)
    except BaseException:
        import traceback
        traceback.print_exc()
        record=session.root/("suite.json" if SUITE else "session.json")
        if record.exists():
            failed=read_json(record);print("Failure:",failed.get("error"),"· stage:",failed.get("stage"),"· successful:",failed.get("progress"),"· full records:",record)
        raise
else:
    print("Training not launched. Set RUN_TRAINING=True deliberately.")


## 7 · Export, verify and persist

No training. Inference export contains all actual mesh modules; it needs no dataset or A1 encoder. Diagnostic/resumable manifests include source, data recipe/splits/scales, all available stage state/history/preview and strict dependency hashes. Frozen modules have one canonical prerequisite, not redundant stage copies. Content-addressed persistent objects deduplicate identical source/dependencies. Preserve all accepted snapshots and user results.

Download an inference ZIP to the Mac, extract it and import `export/model.pt` through the playground's checkpoint picker or `pdeflow mesh import`. Diagnostic and resumable ZIPs serve different purposes. A dynamics-only checkpoint is not advertised as a trained inverse model.

In [ ]:
#@title 7 · Save and verify (does not train)
from pdeflow.mesh.artifacts import verify,archive
SAVE_AND_VERIFY = True
DOWNLOAD_ARTIFACT = "none" # none / inference / diagnostic / resumable; one candidate only
if SAVE_AND_VERIFY:
    if SUITE:
        for candidate in session.inspect()['plan']['order']:
            root=session.root/'runs'/candidate
            if (root/'session.json').exists():print(candidate,MeshSession.reopen(root).finalize())
        print("Suite snapshot:",session.sync('manual-final'))
    elif (session.root/'session.json').exists():
        saved=session.finalize();print(json.dumps(saved,indent=2))
        for kind in ('inference','diagnostic','resumable'):print(kind,verify(session.root,kind)['available'])
        if DOWNLOAD_ARTIFACT!='none':
            from google.colab import files
            files.download(str(archive(session.root,DOWNLOAD_ARTIFACT)))
    else:print("No training was launched; source/config will be persisted before the first update.")


## 8 · Immutable shared comparison

Generate validation cases once for development; keep test untouched until a predeclared final comparison. All candidates and optional legacy use the same actual sensors/noise/coordinates/times and paired base-noise controls. M1 is evaluated only on its initial task. No truth-based best-of-N selection. The report separates per-sample and ensemble-mean physical errors, exact/held sensors, parameters, CRPS/energy/coverage, spatial/temporal dependence, physics and actual costs. Tiny ensembles are not calibrated posteriors. Correct pairing is not claimed better until measured.

Equal examples and wall cost are reported; equal-wall accuracy is unavailable without matched wall-time exports. Capacity labels do not imply equal parameter count. From-scratch and warm-start experiments must be compared separately.

In [ ]:
#@title 8 · Optional cases and comparison (no training)
from pdeflow.mesh.evaluation import create_cases,compare
CREATE_CASES = False
RUN_COMPARISON = False
CASES_DIRECTORY = "/content/pdeflow-mesh-comparison/cases"
COMPARISON_DIRECTORY = "/content/pdeflow-mesh-comparison/report"
CHECKPOINTS = {} # {"M1":".../export/model.pt", "legacy":".../flow/inference.pt"}
if CREATE_CASES:
    science=MeshConfig(**next(iter(session.inspect()['plan']['configs'].values()))) if SUITE else session.c
    print(create_cases(science,CASES_DIRECTORY,count=8,split='validation'))
if RUN_COMPARISON:
    if not CHECKPOINTS:raise ValueError("Supply explicitly trained exports; no random models are substituted")
    compare(CHECKPOINTS,CASES_DIRECTORY,COMPARISON_DIRECTORY,device=DEVICE,particles=8,steps=16)
    print("Report:",Path(COMPARISON_DIRECTORY)/"comparison.json")
